# Scenario 2 (Optimized) — Multi-Step Agent with Central Router

## What evolves from Scenario 1?

| Scenario 1 | Scenario 2 |
|---|---|
| 2 sources (RAG) | RAG + CSV + Web Search |
| Simple routing (1 step) | **Multi-step** routing (loop) |
| One action per query | **Multiple** actions per query |
| Read-only | Read **and write** (CSV) |

## New LangGraph Concepts

| Concept | Description |
|---|---|
| **Central router node** | A hub node that centralizes all routing decisions |
| **Loop in the graph** | The agent can pass through multiple nodes before responding |
| **Tool composition** | A single question may require multiple sources |
| **Data writing** | The agent registers new tickets in the CSV |

## Flow Architecture (hub-and-spoke)

```
                          +-> [Search FAQ]        -+
                          +-> [Query CSV]          -+
[START] -> [Analyze] -> [ROUTER] <-----------------+
                          +-> [Search Web]        -> [ROUTER]
                          +-> [Register Ticket]    -+
                          +-> [Generate Response] -> [END]
```

The **Router** is a central node: it receives the flow after each action and decides the next step.
This produces a clean diagram with a star topology.

## Step 1 — Imports and Configuration

In [ ]:
import os, json
from datetime import date
from dotenv import load_dotenv

load_dotenv()

import pandas as pd
from shared.llm import get_llm
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_databricks import DatabricksEmbeddings
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

llm = get_llm()
print("Dependencies loaded!")

## Step 2 — Preparing the Tools

We will prepare **4 tools** the agent can use:
1. **RAG** — search the FAISS stores (reusing Scenario 1)
2. **CSV Query** — read existing tickets
3. **Web Search** — search the internet via DuckDuckGo
4. **CSV Write** — create a new ticket

In [ ]:
# -- 2.1  Vector stores (same as Scenario 1) --
def create_vector_store(file_path):
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,
                                              separators=["\n## ", "\n### ", "\n\n", "\n", " "])
    chunks = splitter.split_documents(docs)
    embeddings = DatabricksEmbeddings(endpoint=os.getenv("DATABRICKS_EMBEDDINGS_ENDPOINT", "databricks-bge-large-en"))
    return FAISS.from_documents(chunks, embeddings)

policies_store = create_vector_store("data/kb/store_policies.md")
products_store = create_vector_store("data/kb/product_faq.md")
print("Vector stores ready")


# -- 2.2  CSV Query --
CSV_PATH = "data/csv/tickets.csv"

def query_csv(filter_text: str) -> str:
    """Searches tickets in the CSV that contain the filter text."""
    df = pd.read_csv(CSV_PATH)
    mask = df.apply(lambda row: filter_text.lower() in str(row).lower(), axis=1)
    results = df[mask]
    if results.empty:
        return "No tickets found for this filter."
    return results.to_string(index=False)


# -- 2.3  Register new ticket --
def register_ticket(customer, email, category, description, product="", priority="Medium") -> str:
    """Adds a new ticket to the CSV."""
    df = pd.read_csv(CSV_PATH)
    new_id = int(df["id"].max()) + 1
    new_row = {
        "id": new_id,
        "customer": customer,
        "email": email,
        "open_date": date.today().isoformat(),
        "category": category,
        "description": description,
        "status": "Open",
        "priority": priority,
        "product": product,
        "resolution": "",
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    return f"Ticket #{new_id} registered successfully!"


# -- 2.4  Web search --
web_search = DuckDuckGoSearchRun()

def search_web(query: str) -> str:
    """Searches the internet using DuckDuckGo."""
    try:
        return web_search.invoke(query)
    except Exception as e:
        return f"Web search error: {e}"


print("Tools ready: RAG, CSV (read/write), Web Search")

## Step 3 — Defining the Multi-Step State

The state is now richer. Notice the `actions_completed` field:

```python
actions_completed: Annotated[list[str], add]
```

The `Annotated[..., add]` tells LangGraph to **accumulate** (append) instead of replacing.
So each node can return `{"actions_completed": ["faq"]}` and the value is **added** to the existing list.

In [ ]:
class MultiStepState(TypedDict):
    """Multi-step agent state.

    The actions_completed field uses Annotated[list, add] —
    each node ACCUMULATES items instead of overwriting.
    """
    question: str
    intent: dict                              # which actions are needed
    faq_context: str                          # RAG context
    csv_data: str                             # CSV query result
    web_result: str                           # web search result
    ticket_created: str                       # info about created ticket
    actions_completed: Annotated[list, add]   # completed actions (accumulates!)
    response: str                             # final response

print("State defined — MultiStepState with 8 fields")
print("   actions_completed uses Annotated[list, add] for accumulation")

## Step 4 — Node Functions

### Node 1: Analyze Intent
The LLM analyzes the question and returns a JSON indicating which tools are needed.

### Action Nodes: search_faq, query_csv, search_web, register_ticket
Each executes its task and marks the action as completed.

### Router Node (central hub)
Pass-through node that serves as the decision point. The routing logic is in the `decide_next` function, used as a conditional edge from the router. It checks which actions are still pending and routes to the next one — or to the final response.

### Final Node: generate_response
Consolidates everything and generates the response.

In [ ]:
def analyze_intent(state: MultiStepState) -> dict:
    """Uses the LLM to understand what the customer needs."""

    question = state["question"]
    print(f"\nAnalyzing intent: '{question}'")

    prompt = f"""Analyze the request from a TechMart customer (electronics store).
Determine which actions are needed to respond.

Return ONLY a valid JSON (no markdown) with these boolean fields:
- "needs_faq": true if it needs to query policies/product FAQ
- "needs_csv": true if it needs to query existing ticket records
- "needs_web": true if it needs to search for information on the internet
- "needs_registration": true if it needs to create a new ticket/record
- "summary": brief summary of the request
- "csv_filter": search term for the CSV (if needs_csv is true, otherwise "")
- "web_query": search term for the web (if needs_web is true, otherwise "")
- "registration_data": object with customer/email/category/description/product if needs_registration, otherwise null

Request: {question}"""

    resp = llm.invoke(prompt)
    try:
        intent = json.loads(resp.content)
    except json.JSONDecodeError:
        intent = {"needs_faq": True, "needs_csv": False,
                  "needs_web": False, "needs_registration": False,
                  "summary": question}

    flags = [k for k in ("needs_faq", "needs_csv", "needs_web", "needs_registration")
             if intent.get(k)]
    print(f"   Required actions: {flags}")
    return {"intent": intent}


print("analyze_intent node defined")

In [ ]:
def node_search_faq(state: MultiStepState) -> dict:
    """Searches context in the FAQ and policies stores."""
    print("   Executing: Search FAQ/RAG...")
    question = state["question"]
    docs_pol = policies_store.similarity_search(question, k=2)
    docs_prod = products_store.similarity_search(question, k=2)
    all_docs = docs_pol + docs_prod
    context = "\n\n".join(d.page_content for d in all_docs)
    print(f"   {len(all_docs)} chunks retrieved")
    return {"faq_context": context, "actions_completed": ["faq"]}


def node_query_csv(state: MultiStepState) -> dict:
    """Queries existing tickets in the CSV."""
    filter_text = state.get("intent", {}).get("csv_filter", state["question"])
    print(f"   Executing: Query CSV (filter: '{filter_text}')...")
    result = query_csv(filter_text)
    print("   CSV query completed")
    return {"csv_data": result, "actions_completed": ["csv"]}


def node_search_web(state: MultiStepState) -> dict:
    """Searches for information on the internet."""
    query = state.get("intent", {}).get("web_query", state["question"])
    print(f"   Executing: Web search (query: '{query}')...")
    result = search_web(query)
    print("   Web search completed")
    return {"web_result": result, "actions_completed": ["web"]}


def node_register_ticket(state: MultiStepState) -> dict:
    """Registers a new ticket in the CSV."""
    print("   Executing: Register new ticket...")
    data = state.get("intent", {}).get("registration_data", {})
    if data:
        msg = register_ticket(
            customer=data.get("customer", "Customer not provided"),
            email=data.get("email", "not@provided.com"),
            category=data.get("category", "General"),
            description=data.get("description", state["question"]),
            product=data.get("product", ""),
            priority=data.get("priority", "Medium"),
        )
    else:
        msg = register_ticket("Customer", "customer@email.com", "General", state["question"])
    print(f"   {msg}")
    return {"ticket_created": msg, "actions_completed": ["registration"]}


print("Action nodes defined: search_faq, query_csv, search_web, register_ticket")

In [ ]:
def node_router(state: MultiStepState) -> dict:
    """Central hub node — the decision logic is in the conditional edge.

    This node exists to centralize routing. It does not modify
    the state, only prints progress and serves as a convergence
    point for all actions.
    """
    completed = state.get("actions_completed", [])
    print(f"   Router: completed actions = {completed}")
    return {}


def decide_next(state: MultiStepState) -> str:
    """Routing function — used as the conditional edge from the router node.

    Iterates through the intent flags and checks if the
    corresponding action has been completed. If not, routes to it.
    If all are done, goes to generate_response.
    """
    intent = state.get("intent", {})
    completed = state.get("actions_completed", [])

    if intent.get("needs_faq") and "faq" not in completed:
        return "search_faq"
    if intent.get("needs_csv") and "csv" not in completed:
        return "query_csv"
    if intent.get("needs_web") and "web" not in completed:
        return "search_web"
    if intent.get("needs_registration") and "registration" not in completed:
        return "register_ticket"
    return "generate_response"


print("Router node + decision function defined")

In [ ]:
def node_generate_response(state: MultiStepState) -> dict:
    """Consolidates all information and generates the final response."""
    print("   Generating final response...")

    parts = []
    if state.get("faq_context"):
        parts.append(f"FAQ/Policy information:\n{state['faq_context']}")
    if state.get("csv_data"):
        parts.append(f"Ticket data:\n{state['csv_data']}")
    if state.get("web_result"):
        parts.append(f"Web information:\n{state['web_result']}")
    if state.get("ticket_created"):
        parts.append(f"Registration completed:\n{state['ticket_created']}")

    full_context = "\n\n---\n\n".join(parts) if parts else "No information collected."

    prompt = f"""You are a virtual assistant for TechMart (electronics store).
Based on the collected information, respond to the customer's request.
Be clear, polite, and thorough.

Collected information:
{full_context}

Customer request: {state['question']}

Response:"""

    resp = llm.invoke(prompt)
    print("   Response generated!")
    return {"response": resp.content}


print("generate_response node defined")

## Step 5 — Building the Multi-Step Graph (Optimized Version)

In this version, we create a **central router node** (hub-and-spoke):

- Each action node has a **fixed edge** back to the router
- Only the router has **conditional edges** outbound
- Result: clean diagram with ~10 edges instead of ~25

Compare with the original version (`scenario_2_multistep.ipynb`) where each action node had 5 conditional edges — same logic, but a much more cluttered diagram.

In [ ]:
ROUTE_MAP = {
    "search_faq":        "search_faq",
    "query_csv":         "query_csv",
    "search_web":        "search_web",
    "register_ticket":   "register_ticket",
    "generate_response": "generate_response",
}

graph = StateGraph(MultiStepState)

graph.add_node("analyze_intent",    analyze_intent)
graph.add_node("router",            node_router)
graph.add_node("search_faq",        node_search_faq)
graph.add_node("query_csv",         node_query_csv)
graph.add_node("search_web",        node_search_web)
graph.add_node("register_ticket",   node_register_ticket)
graph.add_node("generate_response", node_generate_response)

graph.add_edge(START, "analyze_intent")
graph.add_edge("analyze_intent", "router")

graph.add_conditional_edges("router", decide_next, ROUTE_MAP)

graph.add_edge("search_faq",       "router")
graph.add_edge("query_csv",        "router")
graph.add_edge("search_web",       "router")
graph.add_edge("register_ticket",  "router")

graph.add_edge("generate_response", END)

app = graph.compile()

print("Multi-step graph compiled (hub-and-spoke version)!")

from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

## Step 6 — Testing Different Scenarios

Let's test questions that require **different combinations** of tools:

| Question | Expected Tools |
|---|---|
| Simple FAQ | RAG |
| Ticket status | CSV |
| External information | Web Search |
| New ticket | CSV (write) |
| Complex question | RAG + CSV |

In [ ]:
# Test 1: Simple FAQ (RAG only)
print("=" * 70)
print("TEST 1 — Simple FAQ")
print("=" * 70)
r = app.invoke({"question": "What is the deadline to return a product?", "actions_completed": []})
print(f"\nRESPONSE:\n{r['response']}")
print(f"\nActions completed: {r['actions_completed']}")

In [ ]:
# Test 2: Ticket query (CSV)
print("=" * 70)
print("TEST 2 — Ticket query from CSV")
print("=" * 70)
r = app.invoke({"question": "What is the status of Maria Silva's ticket?", "actions_completed": []})
print(f"\nRESPONSE:\n{r['response']}")
print(f"\nActions completed: {r['actions_completed']}")

In [ ]:
# Test 3: Complex question (multiple sources)
print("=" * 70)
print("TEST 3 — Complex question (multiple sources)")
print("=" * 70)
r = app.invoke({
    "question": "My TechBook Essentials has a flickering screen. "
                "What is the warranty policy? Is there already a similar ticket?",
    "actions_completed": [],
})
print(f"\nRESPONSE:\n{r['response']}")
print(f"\nActions completed: {r['actions_completed']}")

In [ ]:
# Test 4: Register new ticket (CSV write)
print("=" * 70)
print("TEST 4 — Register new ticket")
print("=" * 70)
r = app.invoke({
    "question": "I'm Pedro Henrique (pedro.h@email.com). "
                "My TechView 27 QHD monitor arrived with a dead pixel. "
                "I want to open a ticket.",
    "actions_completed": [],
})
print(f"\nRESPONSE:\n{r['response']}")
print(f"\nActions completed: {r['actions_completed']}")

In [ ]:
# Verify that the ticket was actually written to the CSV
print("\nLatest CSV records:")
df = pd.read_csv("data/csv/tickets.csv")
print(df.tail(3).to_string(index=False))


## Summary

| LangGraph Concept | What We Learned |
|---|---|
| **Central router node** | Hub-and-spoke: all decisions at a single point |
| **Conditional edges with loop** | The router redirects until all actions are completed |
| **Annotated[list, add]** | Accumulate data between iterations without overwriting |
| **Tool composition** | A single question can use FAQ + CSV + Web |
| **Data writing** | The agent modifies external state (CSV) |

### Original vs Optimized Version

| Aspect | Original | Optimized |
|---|---|---|
| Routing | Conditional edges on each action node | Conditional edges only on the router |
| Diagram edges | ~25 | ~10 |
| Logic | Identical | Identical |
| Diagram | Cluttered (many crossing arrows) | Clean (star topology) |